In [ ]:
# Realizar a busca de pares e coleta de dados fotometricos em cada pasta
vezes_repetido = 2

# Backend interativo recomendado para o VS Code
%matplotlib widget

for i in range(vezes_repetido):

    pasta_reduced_estrela_i = f"{pasta_reduced_estrela}/Obs_{i+1}_{NOME_ESTRELA}"
        
    todos_arquivos = glob.glob(os.path.join(pasta_reduced_estrela_i, 'red_*.fits'))

    if not todos_arquivos:
        raise ValueError("Nenhum arquivo 'red_*.fits' encontrado.")

    arquivos_por_filtro = defaultdict(list)
    for arq in todos_arquivos:
        nome = os.path.basename(arq)
        partes = nome.split('_')
        if len(partes) >= 5:
            filtro = partes[2]
            arquivos_por_filtro[filtro].append(arq)

    print("=== PIPELINE BLINDADO DE FOTOMETRIA BEACON ===")
    print(f"Filtros identificados: {list(arquivos_por_filtro.keys())}\n")

    # Loop por Filtro
    for filtro, arquivos in arquivos_por_filtro.items():
        arquivos = sorted(arquivos)
        total_arq = len(arquivos)
        
        print(f"\n{'='*60}")
        print(f"PROCESSANDO FILTRO: '{filtro.upper()}' ({total_arq} imagens)")
        print(f"{'='*60}")
        
        imagem_referencia = arquivos[0]
        data_ref = fits.getdata(imagem_referencia)
        data_ref = np.squeeze(data_ref)
        data_ref = np.nan_to_num(data_ref, nan=np.nanmedian(data_ref), posinf=0, neginf=0)

        # --- SELEÇÃO INTERATIVA POR CLIQUE DO MOUSE (SEM TRAVAR O VIZUALIZADOR) ---
        print(f"\n👉 [FILTRO {filtro.upper()}] Instruções:")
        print("   1. Clique na Estrela Principal (Feixe P)")
        print("   2. Clique na Estrela Secundária (Feixe S)")
        print("   3. Pressione ENTER no campo do terminal após fazer os 2 cliques.")

        pontos = []
        def onclick(event):
            if event.xdata is not None and event.ydata is not None:
                pontos.append((event.xdata, event.ydata))
                rotulo = "P (Principal)" if len(pontos) == 1 else ("S (Secundário)" if len(pontos) == 2 else f"Clique {len(pontos)}")
                print(f"   📍 Capturado {rotulo}: X={event.xdata:.1f}, Y={event.ydata:.1f}")

        fig, ax = plt.subplots(figsize=(7, 7))
        vmin = np.median(data_ref)
        vmax = np.percentile(data_ref, 99.5)
        ax.imshow(data_ref, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
        ax.set_title(f"CLIQUE INTERATIVO - Filtro {filtro.upper()}\n1º clique: Principal (P)  |  2º clique: Secundário (S)", fontsize=11, color='navy')
        
        # Activa a escuta de cliques do rato
        cid = fig.canvas.mpl_connect('button_press_event', onclick)
        plt.show()

        # Aguarda a confirmação do utilizador no terminal
        input("\n--- Faz os 2 cliques na imagem acima e depois pressiona [ENTER] aqui para continuar ---")
        fig.canvas.mpl_disconnect(cid)
        plt.close(fig)

        if len(pontos) < 2:
            print(f"⚠️ [AVISO] Menos de 2 cliques registados ({len(pontos)}). A saltar filtro {filtro.upper()}...")
            continue

        # Extrai coordenadas dos 2 primeiros cliques
        (x_click_p, y_click_p), (x_click_s, y_click_s) = pontos[:2]

        # Refina a posição para o pico exato de brilho via centróide
        x_prin_ref, y_prin_ref = rastrear_centroide_ancorado(data_ref, x_click_p, y_click_p, janela=janela_busca)
        x_sec_ref, y_sec_ref = rastrear_centroide_ancorado(data_ref, x_click_s, y_click_s, janela=janela_busca)

        dx_rigido = x_sec_ref - x_prin_ref
        dy_rigido = y_sec_ref - y_prin_ref

        # --- Verificação humana ---
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
        fig.suptitle(f"Confirmação: Filtro {filtro.upper()} | Ref: {os.path.basename(imagem_referencia)}")

        ax1.imshow(data_ref, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
        ax1.set_title(f"P: Principal ({x_prin_ref:.1f}, {y_prin_ref:.1f})")
        ax1.add_patch(Circle((x_prin_ref, y_prin_ref), raio_abertura, color='lime', fill=False, lw=2))
        ax1.add_patch(Circle((x_prin_ref, y_prin_ref), r_in_sky, color='yellow', fill=False, ls=':'))
        ax1.add_patch(Circle((x_prin_ref, y_prin_ref), r_out_sky, color='yellow', fill=False, ls=':'))
        ax1.set_xlim(x_prin_ref - 25, x_prin_ref + 25)
        ax1.set_ylim(y_prin_ref - 25, y_prin_ref + 25)

        ax2.imshow(data_ref, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
        ax2.set_title(f"S: Secundário ({x_sec_ref:.1f}, {y_sec_ref:.1f})")
        ax2.add_patch(Circle((x_sec_ref, y_sec_ref), raio_abertura, color='red', fill=False, lw=2))
        ax2.add_patch(Circle((x_sec_ref, y_sec_ref), r_in_sky, color='yellow', fill=False, ls=':'))
        ax2.add_patch(Circle((x_sec_ref, y_sec_ref), r_out_sky, color='yellow', fill=False, ls=':'))
        ax2.set_xlim(x_sec_ref - 25, x_sec_ref + 25)
        ax2.set_ylim(y_sec_ref - 25, y_sec_ref + 25)

        plt.tight_layout()
        plt.show()

        confirma = input(f"Filtro {filtro.upper()}: As estrelas e anéis de céu estão corretos? (S/N): ").upper()
        if confirma != 'S' and confirma != '':
            continue

        # --- EXTRAÇÃO COM SUBTRAÇÃO DE CÉU E VÍNCULO RÍGIDO ---
        arquivo_saida = os.path.join(pasta_reduced_estrela_i, f'fotometria_filtro_{filtro}.csv')

        with open(arquivo_saida, 'w') as f_out:
            f_out.write("Arquivo,Posicao,Angulo,X_Prin,Y_Prin,Fluxo_Prin,Pico_Prin,X_Sec,Y_Sec,Fluxo_Sec,Pico_Sec\n")
            
            fluxo_p_ref = None
            fluxo_s_ref = None
            ignorar_alertas = False

            for i, arq in enumerate(arquivos):
                nome = os.path.basename(arq)
                try:
                    hdr = fits.getheader(arq)
                    lamina_raw = str(hdr.get('LAMINA', 'L0')).strip().upper()
                    angulo_psi = mapa_lamina_graus.get(lamina_raw, 0.0)
                    posicao_int = int(angulo_psi / 22.5)
                except Exception as e:
                    posicao_int = 0
                    angulo_psi = 0.0
                
                dados = fits.getdata(arq)
                dados = np.squeeze(dados)
                dados = np.nan_to_num(dados, nan=np.nanmedian(dados), posinf=0, neginf=0)
                
                # 1. Rastreia o feixe principal
                x_prin, y_prin = rastrear_centroide_ancorado(dados, x_prin_ref, y_prin_ref, janela=janela_busca)
                
                # 2. Vínculo rígido do feixe secundário
                x_sec = x_prin + dx_rigido
                y_sec = y_prin + dy_rigido
                
                # --- MEDIÇÃO DO PICO DE CONTAGENS (PARA DETECTAR SATURAÇÃO) ---
                h, w = dados.shape
                xi_p, yi_p = int(round(x_prin)), int(round(y_prin))
                pico_p = np.max(dados[max(0, yi_p-3):min(h, yi_p+4), max(0, xi_p-3):min(w, xi_p+4)])

                xi_s, yi_s = int(round(x_sec)), int(round(y_sec))
                pico_s = np.max(dados[max(0, yi_s-3):min(h, yi_s+4), max(0, xi_s-3):min(w, xi_s+4)])
                
                # 3. Fotometria com Anel de Céu
                aper_p = CircularAperture((x_prin, y_prin), r=raio_abertura)
                ann_p = CircularAnnulus((x_prin, y_prin), r_in=r_in_sky, r_out=r_out_sky)
                phot_p = aperture_photometry(dados, [aper_p, ann_p])
                
                aper_s = CircularAperture((x_sec, y_sec), r=raio_abertura)
                ann_s = CircularAnnulus((x_sec, y_sec), r_in=r_in_sky, r_out=r_out_sky)
                phot_s = aperture_photometry(dados, [aper_s, ann_s])
                
                bkg_p_pixel = phot_p['aperture_sum_1'][0] / area_ann
                fluxo_p = phot_p['aperture_sum_0'][0] - (bkg_p_pixel * area_aper)
                
                bkg_s_pixel = phot_s['aperture_sum_1'][0] / area_ann
                fluxo_s = phot_s['aperture_sum_0'][0] - (bkg_s_pixel * area_aper)
                
                if i == 0:
                    fluxo_p_ref = max(fluxo_p, 100.0)
                    fluxo_s_ref = max(fluxo_s, 100.0)

                # --- LÓGICA DO ALERTA INTELIGENTE ---
                desvio_posicao = np.sqrt((x_prin - x_prin_ref)**2 + (y_prin - y_prin_ref)**2)
                queda_fluxo_p = fluxo_p < (0.50 * fluxo_p_ref)
                queda_fluxo_s = fluxo_s < (0.50 * fluxo_s_ref)
                queda_critica = (fluxo_p < 0.20 * fluxo_p_ref) or (fluxo_s < 0.20 * fluxo_s_ref)
                deslocado_critico = desvio_posicao > 8.0

                ha_anomalia = (queda_critica or deslocado_critico)

                if ha_anomalia and not ignorar_alertas:
                    motivos = []
                    if deslocado_critico: motivos.append(f"Pulo de Posição ({desvio_posicao:.1f}px)")
                    if queda_fluxo_p or queda_fluxo_s: motivos.append("Queda de Sinal (>50% NUVEM/PERDA)")
                    motivo_str = " | ".join(motivos)
                    
                    print(f"\n⚠️  [ALERTA NA IMAGEM {i+1}/{total_arq}] {nome}")
                    print(f"   Motivo: {motivo_str}")
                    
                    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
                    fig.suptitle(f"⚠️ ALERTA: {motivo_str}\n{nome} [{i+1}/{total_arq}]", color='red', fontweight='bold')
                    
                    vmin, vmax = np.median(dados), np.percentile(dados, 99.5)
                    
                    ax1.imshow(dados, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
                    ax1.add_patch(Circle((x_prin, y_prin), raio_abertura, color='lime', fill=False, lw=2))
                    ax1.add_patch(Circle((x_prin, y_prin), r_in_sky, color='yellow', fill=False, ls=':'))
                    ax1.add_patch(Circle((x_prin, y_prin), r_out_sky, color='yellow', fill=False, ls=':'))
                    ax1.set_xlim(x_prin - 25, x_prin + 25)
                    ax1.set_ylim(y_prin - 25, y_prin + 25)
                    ax1.set_title(f"Feixe P: Fluxo={fluxo_p:.0f} | Pico={pico_p:.0f}")
                    
                    ax2.imshow(dados, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
                    ax2.add_patch(Circle((x_sec, y_sec), raio_abertura, color='red', fill=False, lw=2))
                    ax2.add_patch(Circle((x_sec, y_sec), r_in_sky, color='yellow', fill=False, ls=':'))
                    ax2.add_patch(Circle((x_sec, y_sec), r_out_sky, color='yellow', fill=False, ls=':'))
                    ax2.set_xlim(x_sec - 25, x_sec + 25)
                    ax2.set_ylim(y_sec - 25, y_sec + 25)
                    ax2.set_title(f"Feixe S: Fluxo={fluxo_s:.0f} | Pico={pico_s:.0f}")
                    
                    plt.tight_layout()
                    plt.show()
                    
                    print("Opções de ação:\n [ENTER/S] Aceitar e Continuar\n [R] Recalibrar Posição Âncora\n [M] Mudar Referência de Fluxo\n [C] Continuar sem mais alertas")
                    resp = input("Escolha uma opção (ENTER/S/R/M/C): ").upper().strip()
                    if resp == 'R':
                        x_prin_ref = float(input(f"Novo X âncora (Atual: {x_prin:.1f}): "))
                        y_prin_ref = float(input(f"Novo Y âncora (Atual: {y_prin:.1f}): "))
                        x_prin, y_prin = rastrear_centroide_ancorado(dados, x_prin_ref, y_prin_ref, janela=janela_busca)
                        x_sec, y_sec = x_prin + dx_rigido, y_prin + dy_rigido
                    elif resp == 'M':
                        fluxo_p_ref, fluxo_s_ref = max(fluxo_p, 100.0), max(fluxo_s, 100.0)
                    elif resp == 'C':
                        ignorar_alertas = True

                if i % 50 == 0 or i == total_arq - 1:
                    print(f"[{i+1}/{total_arq}] {nome} | Pos {posicao_int} ({angulo_psi}°) -> F_Prin={fluxo_p:.0f} (Pico={pico_p:.0f}) | F_Sec={fluxo_s:.0f} (Pico={pico_s:.0f})")
                
                f_out.write(f"{nome},{posicao_int},{angulo_psi:.1f},{x_prin:.3f},{y_prin:.3f},{fluxo_p:.3f},{pico_p:.1f},{x_sec:.3f},{y_sec:.3f},{fluxo_s:.3f},{pico_s:.1f}\n") 
                              
        print(f"✅ Filtro {filtro.upper()} finalizado e salvo em {arquivo_saida}!")

    print("\n=== RE-EXTRAÇÃO CONCLUÍDA COM SUCESSO! ===")

In [ ]:
# Realizar a busca de pares e coleta de dados fotometricos em cada pasta

%matplotlib inline

# ====================================================================
# FUNÇÃO 1: RECALIBRAÇÃO DE ÂNCORA (CONFIRMAÇÃO VIA ENTER / S)
# ====================================================================
def recalibrar_ancora_cv2(data_frame, x_atual, y_atual, janela_busca):
    h, w = data_frame.shape
    data_clean = np.nan_to_num(data_frame, nan=np.nanmedian(data_frame), posinf=0, neginf=0)
    
    vmin = np.median(data_clean)
    vmax = np.percentile(data_clean, 99.5)
    if vmax <= vmin:
        vmax = vmin + 1.0
    
    img_norm = np.clip((data_clean - vmin) / (vmax - vmin) * 255, 0, 255).astype(np.uint8)
    img_flip = cv2.flip(img_norm, 0)
    img_bgr = cv2.cvtColor(img_flip, cv2.COLOR_GRAY2BGR)
    
    ponto_novo = []
    
    # Marca a posição anterior (incorreta) com ponto vermelho
    yp_cv, xp_cv = int(round((h - 1) - y_atual)), int(round(x_atual))
    cv2.circle(img_bgr, (xp_cv, yp_cv), 2, (0, 0, 255), -1)

    def mouse_cb(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(ponto_novo) == 0:
            y_fits = (h - 1) - y
            ponto_novo.append((x, y_fits))

    nome_janela_base = "RECALIBRACAO DE ANCORA"
    cv2.namedWindow(nome_janela_base, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(nome_janela_base, 800, 800)
    cv2.setMouseCallback(nome_janela_base, mouse_cb)

    print("\n👉 [RECALIBRAÇÃO] Clique na NOVA posição do Feixe P e aperte [ENTER] ou [S].")

    while True:
        img_draw = img_bgr.copy()
        
        if len(ponto_novo) == 0:
            titulo = "RECALIBRAR -> Clique na nova posicao do Feixe P"
        else:
            xn, yn = ponto_novo[0]
            yn_cv, xn_cv = int(round((h - 1) - yn)), int(round(xn))
            cv2.circle(img_draw, (xn_cv, yn_cv), 2, (0, 255, 0), -1)
            cv2.putText(img_draw, "P", (xn_cv + 3, yn_cv - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 255, 0), 1)
            titulo = "RECALIBRAR -> Aperte [ENTER] para Confirmar | [R] para Refazer"

        cv2.setWindowTitle(nome_janela_base, titulo)
        cv2.imshow(nome_janela_base, img_draw)
        key = cv2.waitKey(30) & 0xFF

        # Tecla 13 = ENTER, ord('s')/ord('S') = S
        if key in [13, ord('s'), ord('S')] and len(ponto_novo) > 0:
            break
        elif key in [ord('r'), ord('R')]:
            ponto_novo.clear()
        elif key in [27, ord('q'), ord('Q')]:
            break

    cv2.destroyAllWindows()
    for _ in range(5): cv2.waitKey(1)

    if len(ponto_novo) > 0:
        xn, yn = ponto_novo[0]
        return rastrear_centroide_ancorado(data_clean, xn, yn, janela=janela_busca)
    
    return x_atual, y_atual


# ====================================================================
# FUNÇÃO 2: SELEÇÃO E CONFIRMAÇÃO DO PAR (CONFIRMAÇÃO VIA ENTER / S)
# ====================================================================
def selecionar_e_confirmar_par_cv2(data_ref, filtro_nome, raio_abertura, r_in_sky, r_out_sky, janela_busca):
    h, w = data_ref.shape
    vmin, vmax = np.median(data_ref), np.percentile(data_ref, 99.5)
    
    img_norm = np.clip((data_ref - vmin) / (vmax - vmin) * 255, 0, 255).astype(np.uint8)
    img_flip = cv2.flip(img_norm, 0)
    img_bgr = cv2.cvtColor(img_flip, cv2.COLOR_GRAY2BGR)
    
    pontos = []
    x_p_ref, y_p_ref = None, None
    x_s_ref, y_s_ref = None, None
    
    nome_janela = f"SELECAO DE PAR - FILTRO {filtro_nome.upper()}"
    
    def atualizar_tela():
        img_draw = img_bgr.copy()
        
        if len(pontos) == 0:
            titulo = f"FILTRO {filtro_nome.upper()} -> [1/2] Clique no Feixe PRINCIPAL (P)"
        elif len(pontos) == 1:
            titulo = f"FILTRO {filtro_nome.upper()} -> [2/2] Clique no Feixe SECUNDARIO (S)"
            xp, yp_cv = int(pontos[0][0]), int((h - 1) - pontos[0][1])
            cv2.circle(img_draw, (xp, yp_cv), 2, (0, 255, 0), -1)
        elif len(pontos) == 2:
            nonlocal x_p_ref, y_p_ref, x_s_ref, y_s_ref
            x_p_ref, y_p_ref = rastrear_centroide_ancorado(data_ref, pontos[0][0], pontos[0][1], janela=janela_busca)
            x_s_ref, y_s_ref = rastrear_centroide_ancorado(data_ref, pontos[1][0], pontos[1][1], janela=janela_busca)
            
            yp_cv, xp_cv = int(round((h - 1) - y_p_ref)), int(round(x_p_ref))
            ys_cv, xs_cv = int(round((h - 1) - y_s_ref)), int(round(x_s_ref))
            
            # Feixe P (Verde)
            cv2.circle(img_draw, (xp_cv, yp_cv), int(raio_abertura), (0, 255, 0), 1)
            cv2.circle(img_draw, (xp_cv, yp_cv), int(r_in_sky), (0, 255, 255), 1)
            cv2.circle(img_draw, (xp_cv, yp_cv), int(r_out_sky), (0, 255, 255), 1)
            cv2.putText(img_draw, "P", (xp_cv + 4, yp_cv - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 255, 0), 1)
            
            # Feixe S (Vermelho)
            cv2.circle(img_draw, (xs_cv, ys_cv), int(raio_abertura), (0, 0, 255), 1)
            cv2.circle(img_draw, (xs_cv, ys_cv), int(r_in_sky), (0, 255, 255), 1)
            cv2.circle(img_draw, (xs_cv, ys_cv), int(r_out_sky), (0, 255, 255), 1)
            cv2.putText(img_draw, "S", (xs_cv + 4, ys_cv - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 0, 255), 1)
            
            titulo = f"FILTRO {filtro_nome.upper()} -> Teclas: [ENTER] Confirmar | [R] Refazer | [ESC] Pular"

        cv2.setWindowTitle(nome_janela, titulo)
        cv2.imshow(nome_janela, img_draw)

    def mouse_callback(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(pontos) < 2:
            y_fits = (h - 1) - y
            pontos.append((x, y_fits))
            atualizar_tela()

    cv2.namedWindow(nome_janela, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(nome_janela, 800, 800)
    cv2.setMouseCallback(nome_janela, mouse_callback)
    atualizar_tela()

    print(f"\n👉 [FILTRO {filtro_nome.upper()}] Selecione P e S na janela e aperte [ENTER] para confirmar.\n")

    confirmado = False
    while True:
        key = cv2.waitKey(50) & 0xFF
        # Tecla 13 = ENTER, ord('s')/ord('S') = S
        if key in [13, ord('s'), ord('S')] and len(pontos) == 2:
            confirmado = True
            break
        elif key in [ord('r'), ord('R')]:
            pontos.clear()
            atualizar_tela()
            print("   🔄 Cliques limpos. Selecione P e S novamente.")
        elif key in [27, ord('q'), ord('Q')]:
            confirmado = False
            break

    cv2.destroyAllWindows()
    for _ in range(5): cv2.waitKey(1)
        
    if confirmado:
        return x_p_ref, y_p_ref, x_s_ref, y_s_ref
    else:
        return None, None, None, None


# ====================================================================
# LOOP PRINCIPAL DE BUSCA E EXTRAÇÃO FOTOMÉTRICA
# ====================================================================
for i in range(vezes_repetido):

    pasta_reduced_estrela_i = f"{pasta_reduced_estrela}/Obs_{i+1}_{NOME_ESTRELA}"
        
    todos_arquivos = glob.glob(os.path.join(pasta_reduced_estrela_i, 'red_*.fits'))

    if not todos_arquivos:
        print(f"[AVISO] Nenhum arquivo 'red_*.fits' encontrado em {pasta_reduced_estrela_i}.")
        continue

    arquivos_por_filtro = defaultdict(list)
    for arq in todos_arquivos:
        nome = os.path.basename(arq)
        partes = nome.split('_')
        if len(partes) >= 5:
            filtro = partes[2]
            arquivos_por_filtro[filtro].append(arq)

    print("=== PIPELINE BLINDADO DE FOTOMETRIA BEACON ===")
    print(f"Filtros identificados na Obs {i+1}: {list(arquivos_por_filtro.keys())}\n")

    for filtro, arquivos in arquivos_por_filtro.items():
        arquivos = sorted(arquivos)
        total_arq = len(arquivos)
        
        print(f"\n{'='*60}")
        print(f"PROCESSANDO FILTRO: '{filtro.upper()}' ({total_arq} imagens)")
        print(f"{'='*60}")
        
        imagem_referencia = arquivos[0]
        data_ref = fits.getdata(imagem_referencia)
        data_ref = np.squeeze(data_ref)
        data_ref = np.nan_to_num(data_ref, nan=np.nanmedian(data_ref), posinf=0, neginf=0)

        # 1. Seleção e Confirmação Direta no OpenCV
        x_prin_ref, y_prin_ref, x_sec_ref, y_sec_ref = selecionar_e_confirmar_par_cv2(
            data_ref, filtro, raio_abertura, r_in_sky, r_out_sky, janela_busca
        )

        if x_prin_ref is None:
            print(f"⚠️ [AVISO] Seleção ignorada/cancelada. Pulando filtro {filtro.upper()}...")
            continue

        dx_rigido = x_sec_ref - x_prin_ref
        dy_rigido = y_sec_ref - y_prin_ref

        # 2. Extração Fotométrica nos Arquivos
        arquivo_saida = os.path.join(pasta_reduced_estrela_i, f'fotometria_filtro_{filtro}.csv')

        with open(arquivo_saida, 'w') as f_out:
            f_out.write("Arquivo,Posicao,Angulo,X_Prin,Y_Prin,Fluxo_Prin,Pico_Prin,X_Sec,Y_Sec,Fluxo_Sec,Pico_Sec\n")
            
            fluxo_p_ref = None
            fluxo_s_ref = None
            ignorar_alertas = False

            for i_arq, arq in enumerate(arquivos):
                nome = os.path.basename(arq)
                try:
                    hdr = fits.getheader(arq)
                    lamina_raw = str(hdr.get('LAMINA', 'L0')).strip().upper()
                    angulo_psi = mapa_lamina_graus.get(lamina_raw, 0.0)
                    posicao_int = int(angulo_psi / 22.5)
                except Exception:
                    posicao_int, angulo_psi = 0, 0.0
                
                dados = fits.getdata(arq)
                dados = np.squeeze(dados)
                dados = np.nan_to_num(dados, nan=np.nanmedian(dados), posinf=0, neginf=0)
                
                x_prin, y_prin = rastrear_centroide_ancorado(dados, x_prin_ref, y_prin_ref, janela=janela_busca)
                x_sec = x_prin + dx_rigido
                y_sec = y_prin + dy_rigido
                
                h, w = dados.shape
                xi_p, yi_p = int(round(x_prin)), int(round(y_prin))
                pico_p = np.max(dados[max(0, yi_p-3):min(h, yi_p+4), max(0, xi_p-3):min(w, xi_p+4)])

                xi_s, yi_s = int(round(x_sec)), int(round(y_sec))
                pico_s = np.max(dados[max(0, yi_s-3):min(h, yi_s+4), max(0, xi_s-3):min(w, xi_s+4)])
                
                aper_p = CircularAperture((x_prin, y_prin), r=raio_abertura)
                ann_p = CircularAnnulus((x_prin, y_prin), r_in=r_in_sky, r_out=r_out_sky)
                phot_p = aperture_photometry(dados, [aper_p, ann_p])
                
                aper_s = CircularAperture((x_sec, y_sec), r=raio_abertura)
                ann_s = CircularAnnulus((x_sec, y_sec), r_in=r_in_sky, r_out=r_out_sky)
                phot_s = aperture_photometry(dados, [aper_s, ann_s])
                
                bkg_p_pixel = phot_p['aperture_sum_1'][0] / area_ann
                fluxo_p = phot_p['aperture_sum_0'][0] - (bkg_p_pixel * area_aper)
                
                bkg_s_pixel = phot_s['aperture_sum_1'][0] / area_ann
                fluxo_s = phot_s['aperture_sum_0'][0] - (bkg_s_pixel * area_aper)
                
                if i_arq == 0:
                    fluxo_p_ref, fluxo_s_ref = max(fluxo_p, 100.0), max(fluxo_s, 100.0)

                # Alertas Inteligentes
                desvio_posicao = np.sqrt((x_prin - x_prin_ref)**2 + (y_prin - y_prin_ref)**2)
                queda_critica = (fluxo_p < 0.20 * fluxo_p_ref) or (fluxo_s < 0.20 * fluxo_s_ref)
                deslocado_critico = desvio_posicao > 8.0

                if (queda_critica or deslocado_critico) and not ignorar_alertas:
                    print(f"\n⚠️ [ALERTA IMAGEM {i_arq+1}/{total_arq}] {nome}")
                    
                    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
                    vmin, vmax = np.median(dados), np.percentile(dados, 99.5)
                    
                    ax1.imshow(dados, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
                    ax1.add_patch(Circle((x_prin, y_prin), raio_abertura, color='lime', fill=False, lw=2))
                    ax1.set_xlim(x_prin - 25, x_prin + 25)
                    ax1.set_ylim(y_prin - 25, y_prin + 25)
                    ax1.set_title(f"P: Fluxo={fluxo_p:.0f}")
                    
                    ax2.imshow(dados, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
                    ax2.add_patch(Circle((x_sec, y_sec), raio_abertura, color='red', fill=False, lw=2))
                    ax2.set_xlim(x_sec - 25, x_sec + 25)
                    ax2.set_ylim(y_sec - 25, y_sec + 25)
                    ax2.set_title(f"S: Fluxo={fluxo_s:.0f}")
                    
                    plt.tight_layout()
                    plt.show()

                    print("Opções: [ENTER/S] Aceitar | [R] Recalibrar Âncora por Clique | [M] Mudar Ref | [C] Desativar Alertas")
                    resp = input("Escolha (ENTER/S/R/M/C): ").upper().strip()
                    if resp == 'R':
                        x_prin_ref, y_prin_ref = recalibrar_ancora_cv2(dados, x_prin, y_prin, janela_busca)
                        x_prin, y_prin = x_prin_ref, y_prin_ref
                        x_sec, y_sec = x_prin + dx_rigido, y_prin + dy_rigido
                        print(f"-> Âncora recalibrada para X={x_prin_ref:.1f}, Y={y_prin_ref:.1f}")
                    elif resp == 'M':
                        fluxo_p_ref, fluxo_s_ref = max(fluxo_p, 100.0), max(fluxo_s, 100.0)
                    elif resp == 'C':
                        ignorar_alertas = True

                if i_arq % 50 == 0 or i_arq == total_arq - 1:
                    print(f"[{i_arq+1}/{total_arq}] {nome} | Pos {posicao_int} ({angulo_psi}°) -> F_Prin={fluxo_p:.0f} | F_Sec={fluxo_s:.0f}")
                
                f_out.write(f"{nome},{posicao_int},{angulo_psi:.1f},{x_prin},{y_prin},{fluxo_p:.3f},{pico_p:.1f},{x_sec},{y_sec},{fluxo_s:.3f},{pico_s:.1f}\n") 
                              
        print(f"✅ Filtro {filtro.upper()} finalizado e salvo em {arquivo_saida}!")

print("\n=== EXTRAÇÃO FOTOMÉTRICA CONCLUÍDA COM SUCESSO! ===")